# QC filtering 03 — tipping point

Does Pal et al.'s per-sample QC remove cells at different rates in BRCA1+/- donors
and premenopausal WT donors? The donor is the unit of replication. This series
answers that in two phases. Phase 1 does not transfer labels onto removed cells.

| Notebook | Question |
|---|---|
| `00` cohort | Which 12 donors are in the comparison, and which analysis choices are frozen before any test? |
| `01` filtering rate | Does the removed fraction, overall and by reason, differ by genotype? |
| `02` Pal reference | Do the authors' own labels reproduce their retained-cell result (P = 0.14)? |
| `03` tipping point | How LP-enriched would BRCA1 removed cells have to be to hide an LP expansion? |

**This notebook scores retained cells with the Lim 2009 signatures already in the repo.**
The score is scanpy's up-minus-down score on every gene, not a highly variable subset.
A cell whose winning score is at most zero is called `other`. Removed cells are not scored.

The baseline is the one-sided donor permutation test of retained LP share
(BRCA1 greater than WT, α = 0.01). The tipping point fills in a hypothetical
LP share for the removed cells: WT removed cells keep that donor's retained share,
and BRCA1 removed cells take k times theirs, capped at 1. k* is the smallest k
at which the same test, on the reconstructed all-cell share, reaches p ≤ 0.01.

The zone ceiling is recomputed here, on these predicted types, with the same
near-threshold band as notebook 02. The Fig 4B cluster enrichment stored by
notebook 02 stays on record and is not the ceiling.

In [ ]:
import logging

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from signals_in_the_noise.analysis.filtering_rate import load_phase1, load_specimen_adatas
from signals_in_the_noise.analysis.lineage import retained_lineage_labels
from signals_in_the_noise.analysis.qc_filtering import output_directory
from signals_in_the_noise.analysis.tipping_point import (
    FIGURE,
    run_tipping_point,
    write_tipping_point_outputs,
)
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

OUTPUT_DIR = output_directory()
OUTPUT_DIR

## 1. Retained-cell labels

Cell and donor tables were written by notebook 00. Scoring uses the cached raw
counts for these 12 specimens and subsets each matrix to the retained barcodes
before scoring. This step is the slow one.

In [ ]:
phase1 = load_phase1(OUTPUT_DIR)
donors = pd.read_csv(OUTPUT_DIR / "donors.csv", index_col=0)
cells = pd.read_csv(OUTPUT_DIR / "cells.csv")
adatas = load_specimen_adatas(donors.index, get_data_path("GSE161529_adata_cache"))
lineage = retained_lineage_labels(cells, adatas, donors)
lineage["shares"]

## 2. Baseline, k*, and the zone

The handoff below is the text to paste back after this notebook runs.

In [ ]:
result = run_tipping_point(
    cells,
    donors,
    phase1,
    lineage["labels"],
    lineage["shares"],
    rng=np.random.default_rng(0),
)
handoff = write_tipping_point_outputs(result, OUTPUT_DIR)
display(result["donors"])
display(Image(filename=str(OUTPUT_DIR / FIGURE)))
display(Markdown(handoff))